# RQ2 · Journal characteristics and impact
### Within a field, what journal-level attributes predict higher publication volume and MeSH breadth, and how has that shifted between 2010 and 2023?

**Design.**
- Unit of analysis: journal × year.
- Sample: journals with ≥ 5 articles/year in the chosen field, 2010-2023.
- Outcomes:
  - `log_articles`  -  log article count in the field
  - `mesh_breadth`  -  number of *distinct* MeSH terms across the year's articles (a proxy for topical scope)
  - `mean_n_authors`  -  mean author-list length (proxy for team size / big-science)
- Regressors of interest: `journal_age` (years since first article of the journal in PubMed), `is_open_access` (heuristic from journal name), year trend.
- Fixed effects: journal + year (two-way).
- SEs: clustered at journal.

**Why care?** Bibliometrics folks debate whether "topic-broad" journals
(Nature, PNAS) crowd out specialist journals in emerging fields, or
whether the specialists grow alongside. Journal FE strips level
differences; year FE removes field-wide expansion; the estimand is
*within-journal deviation from its trend*.

**Compute footprint.** Same pull as RQ1 (100K-200K articles); the panel
collapse is fast. OSG buys us bootstrap SEs and lets us re-run for
multiple field queries in parallel.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ncbi_client import pubmed_query_to_dataframe
from src.regression import fixed_effects_panel


## 1. Pull the field literature (reuse or refresh)

In [ ]:
FIELD_QUERY = '"CRISPR-Cas Systems"[mesh] OR "CRISPR"[tiab]'
lit = pubmed_query_to_dataframe(
    FIELD_QUERY,
    mindate="2010/01/01", maxdate="2023/12/31",
    max_records=100000,
)
print(f"{len(lit):,} articles")


## 2. Collapse to journal × year

In [ ]:
def unique_mesh(series):
    all_terms = set()
    for s in series.dropna():
        for term in s.split(";"):
            if term: all_terms.add(term)
    return len(all_terms)

jy = (
    lit.dropna(subset=["year", "journal"])
    .groupby(["journal", "year"])
    .agg(n_articles=("pmid", "count"),
         mesh_breadth=("mesh_terms", unique_mesh),
         mean_n_authors=("n_authors", "mean"))
    .reset_index()
)
jy = jy[jy["n_articles"] >= 5]
jy["log_articles"] = np.log(jy["n_articles"])
print("journal-year cells:", len(jy), "| journals:", jy["journal"].nunique())
jy.head()


## 3. Build journal-level attributes

In [ ]:
first_year = jy.groupby("journal")["year"].min().rename("journal_first_year")
jy = jy.merge(first_year, on="journal")
jy["journal_age"] = jy["year"] - jy["journal_first_year"]

# Very rough open-access indicator: journal title contains open-access signals.
OA_TOKENS = ["PLoS", "PLOS", "BMC ", "Frontiers in", "eLife", "F1000", "Scientific Reports", "PeerJ", "MDPI", "Cureus"]
jy["is_open_access"] = jy["journal"].apply(lambda s: any(tok in s for tok in OA_TOKENS) if isinstance(s,str) else False).astype(int)

jy.head()


## 4. Descriptive: OA vs subscription over time

In [ ]:
by_year = jy.groupby(["year", "is_open_access"])["n_articles"].sum().unstack(fill_value=0)
by_year.plot(kind="area", stacked=True, figsize=(9,4), alpha=0.7)
plt.ylabel("Total articles"); plt.title("OA vs subscription share of field publications")
plt.legend(["Subscription", "Open access (heuristic)"]); plt.tight_layout()


## 5. Regression  -  volume equation

In [ ]:
res_vol = fixed_effects_panel(
    jy,
    dependent="log_articles",
    regressors=["journal_age", "is_open_access", "mesh_breadth"],
    entity="journal",
    entity_effects=True, time_effects=True,
    name="RQ2  -  log(articles) on journal age + OA + breadth",
)
print(res_vol.as_markdown())


## 6. Regression  -  MeSH breadth equation

In [ ]:
res_breadth = fixed_effects_panel(
    jy,
    dependent="mesh_breadth",
    regressors=["log_articles", "journal_age", "is_open_access"],
    entity="journal",
    entity_effects=True, time_effects=True,
    name="RQ2  -  MeSH breadth on volume + age + OA",
)
print(res_breadth.as_markdown())


In [ ]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
res_vol.coef_table.to_csv("../results/rq2_journal_volume_coefs.csv")
res_breadth.coef_table.to_csv("../results/rq2_journal_breadth_coefs.csv")
